In [23]:
from neo4j import GraphDatabase
import pandas as pd
import random

In [24]:
URI = "bolt://localhost:7687" # URI ZMODYFIKOWANE ZE WZGLĘDU NA POŁĄCZENIE SPOZA KONTENERA Z VSCODEM
USERNAME = "neo4j" 
PASSWORD = "test1234"

## Stworzenie modelu grafu i wypełnienie go danymi

In [25]:
def clear_database(tx):
    tx.run("MATCH (n) DETACH DELETE n")

def load_data(tx):
    tx.run("CREATE (:Category {name: 'Data Science'})")
    tx.run("CREATE (:Category {name: 'Programming'})")
    tx.run("CREATE (:Category {name: 'AI'})")
    tx.run("CREATE (:Category {name: 'Machine Learning'})")
    tx.run("CREATE (:Category {name: 'Web Development'})")
    
    tx.run("CREATE (:Course {id: 1, title: 'Intro to AI'})")
    tx.run("CREATE (:Course {id: 2, title: 'Data Science Basics'})")
    tx.run("CREATE (:Course {id: 3, title: 'Python Programming'})")
    tx.run("CREATE (:Course {id: 4, title: 'Machine Learning 101'})")
    tx.run("CREATE (:Course {id: 5, title: 'Deep Learning Fundamentals'})")
    tx.run("CREATE (:Course {id: 6, title: 'SQL for Data Analysts'})")
    tx.run("CREATE (:Course {id: 7, title: 'Big Data Analytics'})")
    tx.run("CREATE (:Course {id: 8, title: 'Data Visualization with Python'})")
    tx.run("CREATE (:Course {id: 9, title: 'AI Ethics and Fairness'})")
    tx.run("CREATE (:Course {id: 10, title: 'Computer Vision Basics'})")
    
    tx.run("MATCH (c:Course {id: 1}), (cat:Category {name: 'AI'}) CREATE (c)-[:BELONGS_TO]->(cat)")
    tx.run("MATCH (c:Course {id: 2}), (cat:Category {name: 'Data Science'}) CREATE (c)-[:BELONGS_TO]->(cat)")
    tx.run("MATCH (c:Course {id: 3}), (cat:Category {name: 'Programming'}) CREATE (c)-[:BELONGS_TO]->(cat)")
    tx.run("MATCH (c:Course {id: 4}), (cat:Category {name: 'Machine Learning'}) CREATE (c)-[:BELONGS_TO]->(cat)")
    tx.run("MATCH (c:Course {id: 5}), (cat:Category {name: 'AI'}) CREATE (c)-[:BELONGS_TO]->(cat)")
    tx.run("MATCH (c:Course {id: 6}), (cat:Category {name: 'Data Science'}) CREATE (c)-[:BELONGS_TO]->(cat)")
    tx.run("MATCH (c:Course {id: 7}), (cat:Category {name: 'Big Data Analytics'}) CREATE (c)-[:BELONGS_TO]->(cat)")
    tx.run("MATCH (c:Course {id: 8}), (cat:Category {name: 'Data Science'}) CREATE (c)-[:BELONGS_TO]->(cat)")
    tx.run("MATCH (c:Course {id: 9}), (cat:Category {name: 'AI'}) CREATE (c)-[:BELONGS_TO]->(cat)")
    tx.run("MATCH (c:Course {id: 10}), (cat:Category {name: 'Programming'}) CREATE (c)-[:BELONGS_TO]->(cat)")
    
    tx.run("""
        UNWIND range(1, 500) AS id
        CREATE (:User {id: id, name: 'User_' + id})
    """)

    for course_id in range(1, 11):
        for i in range(random.randint(50, 100)):
            user_id = random.randint(1, 500)
            tx.run(f"""
                MATCH (u:User {{id: {user_id}}}), (c:Course {{id: {course_id}}})
                CREATE (u)-[:ENROLLED_IN]->(c)
            """)

    for i in range(200):
        user_id = random.randint(1, 500)
        courses = random.sample(range(1, 11), random.randint(2, 5))
        for course_id in courses:
            tx.run(f"""
                MATCH (u:User {{id: {user_id}}}), (c:Course {{id: {course_id}}})
                CREATE (u)-[:ENROLLED_IN]->(c)
            """)

    for i in range(1000):
        user_id = random.randint(1, 500)
        tx.run(f"""
            MATCH (u:User {{id: {user_id}}}), (c:Course {{id: 1}})
            CREATE (u)-[:ENROLLED_IN]->(c)
        """)

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

with driver.session() as session:
    session.execute_write(clear_database)
    session.execute_write(load_data)

driver.close()


In [26]:
driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

def run_query(query: str, explain_or_profile=False):
    from neo4j import GraphDatabase
    driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
    with driver.session() as session:
        result = session.run(query)

        if explain_or_profile:
            _ = list(result)
            summary = result.consume()
            return summary
        else:
            records = list(result)
            if not records:
                print("No records")
                return
            return pd.DataFrame([r.data() for r in records])



In [27]:
explain_summary = run_query("""
EXPLAIN MATCH (u:User)-[:ENROLLED_IN]->(c:Course)
WHERE c.title = 'Intro to AI'
RETURN u, c
""", explain_or_profile=True)

print(explain_summary)

In [28]:
profile_summary = run_query("""
PROFILE MATCH (u:User)-[:ENROLLED_IN]->(c:Course)
WHERE c.title = 'Intro to AI'
RETURN u, c
""", explain_or_profile=True).profile

print(profile_summary)

{'args': {'GlobalMemory': 64, 'planner-impl': 'IDP', 'Memory': 0, 'string-representation': 'Cypher 5\n\nPlanner COST\n\nRuntime SLOTTED\n\nRuntime version 5.26\n\n+------------------+----+-------------------------------+----------------+------+---------+----------------+------------------------+\n| Operator         | Id | Details                       | Estimated Rows | Rows | DB Hits | Memory (Bytes) | Page Cache Hits/Misses |\n+------------------+----+-------------------------------+----------------+------+---------+----------------+------------------------+\n| +ProduceResults  |  0 | u, c                          |            124 | 1141 |    6846 |              0 |                    0/0 |\n| |                +----+-------------------------------+----------------+------+---------+----------------+------------------------+\n| +Filter          |  1 | u:User                        |            124 | 1141 |    1141 |                |                    0/0 |\n| |                +----+--

In [29]:
driver.close()

## Optymalizacja przez rozdzielenie kursu "Intro to AI" na kursy prowadzone w różnych latach i partycjonowanie użytkowników zapisanych do konkretnego roku

In [30]:
driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

with driver.session() as session:
    session.run("""
    CREATE (:Course {id: 1, title: 'Intro to AI', year: 2023}),
           (:Course {id: 2, title: 'Intro to AI', year: 2024}),
           (:Course {id: 3, title: 'Intro to AI', year: 2025})
    """)

    session.run("""
    UNWIND range(1, 1137) AS id
    MATCH (u:User {id: id})
    WITH u, CASE 
        WHEN id % 3 = 0 THEN 1
        WHEN id % 3 = 1 THEN 2
        ELSE 3
    END AS courseId
    MATCH (c:Course {id: courseId})
    MERGE (u)-[:ENROLLED_IN]->(c)
    """)

driver.close()

In [31]:
driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

In [32]:
explain_optimised_summary = run_query("""
EXPLAIN MATCH (u:User)-[:ENROLLED_IN]->(c:Course)
WHERE c.title = 'Intro to AI' AND c.year = 2023
RETURN u, c
""", explain_or_profile=True)

print(explain_summary)

In [33]:
profile_optimised_summary = run_query("""
PROFILE MATCH (u:User)-[:ENROLLED_IN]->(c:Course)
WHERE c.title = 'Intro to AI' AND c.year = 2023
RETURN u, c
""", explain_or_profile=True).profile

print(profile_optimised_summary)

{'args': {'GlobalMemory': 64, 'planner-impl': 'IDP', 'Memory': 0, 'string-representation': 'Cypher 5\n\nPlanner COST\n\nRuntime SLOTTED\n\nRuntime version 5.26\n\n+------------------+----+---------------------------------------------------+----------------+------+---------+----------------+------------------------+\n| Operator         | Id | Details                                           | Estimated Rows | Rows | DB Hits | Memory (Bytes) | Page Cache Hits/Misses |\n+------------------+----+---------------------------------------------------+----------------+------+---------+----------------+------------------------+\n| +ProduceResults  |  0 | u, c                                              |              8 |  166 |    1162 |              0 |                    0/0 |\n| |                +----+---------------------------------------------------+----------------+------+---------+----------------+------------------------+\n| +Filter          |  1 | u:User                              

In [34]:

driver.close()

## Porównanie przed vs po optymalizacji

In [35]:
def extract_profile_stats(profile):
    args = profile["args"]
    return {
        "DB Hits": args.get("DbHits", 0),
        "Estimated Rows": round(args.get("EstimatedRows", 0), 2),
        "Rows Returned": args.get("Rows", 0),
        "Page Cache Hits": args.get("PageCacheHits", 0),
        "Page Cache Misses": args.get("PageCacheMisses", 0),
        "Total Memory (Bytes)": args.get("GlobalMemory", 0),
    }

before_stats = extract_profile_stats(profile_summary)
after_stats = extract_profile_stats(profile_optimised_summary)

print(f"{'Metryka':<25} {'Przed optymalizacją':>20} {'Po optymalizacji':>20}")
print("-" * 70)
for key in before_stats:
    print(f"{key:<25} {before_stats[key]:>20} {after_stats[key]:>20}")


Metryka                    Przed optymalizacją     Po optymalizacji
----------------------------------------------------------------------
DB Hits                                   6846                 1162
Estimated Rows                           124.1                 8.13
Rows Returned                             1141                  166
Page Cache Hits                              0                    0
Page Cache Misses                            0                    0
Total Memory (Bytes)                        64                   64
